# 05 — ACDC : objet 128², relaxation des échos, rotations et miroirs

Ce notebook intègre le modèle de la V4 dans le pipeline complet du 04 : lecture des
scans, préparation des sous-spins, shim statique, optimisation d'image, réponse
matérielle, contrôles DAC, sauvegarde et export CSV facultatif.

L'objet provient des trois échos natifs 128². Les courants restent communs aux sous-spins
et sont optimisés avec les cartes ACDC 3D. Les limites restent ±3 A par canal,
30 A au total et une réponse équivalente à 99 % en 133 µs.

Les notebooks précédents et les fichiers d'acquisition sont conservés.

In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib, json, sys, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown

PROJECT_ROOT=Path.cwd() if (Path.cwd()/"Experiment").is_dir() else Path.cwd().parent
for folder in (PROJECT_ROOT/"Experiment",PROJECT_ROOT/"MRzero_validation"):
    if str(folder) not in sys.path:sys.path.insert(0,str(folder))
from twin_inputs import read_twin_inputs,prepare_echo_object
from echo_shim_inputs import prepare_echo_shim_inputs
from experiment_echo import EchoSubspinACDCModel
from inplane_geometry import inplane_matrix,FrameReconstruction
from epi_twin import NativeEPITwin
from twin_geometry import compare_magnitude
from acdc_gpu_field import solve_field_gpu
from acdc_gpu import response_matrices
from experiment_optimization import optimize_mrzero_image
from live_optimization import LiveOptimizationPlot
from export_acdc_waveform import quantize_currents,export_waveform
from shim_export_timing import sequence_trigger_timing,waveform_edges_for_origin
from prepare_acdc_inputs import read_scan

INPUT_DIR=PROJECT_ROOT/"Mrzero_simulation/acquisition_synthetic_001"
OUTPUT_DIR=PROJECT_ROOT/"MRzero_validation/results_experiment_echo"/datetime.now().strftime("mrzero05_%Y%m%d_%H%M%S_%f")

## 1. Paramètres à modifier

**Orientation données → bobines** : `ALIGNMENT_ANGLE_DEG` donne l'angle autour de Z,
en degrés positifs de X vers Y. `ALIGNMENT_MIRROR_X` remplace X par −X ;
`ALIGNMENT_MIRROR_Y` remplace Y par −Y. Les miroirs sont appliqués **avant** la rotation.
L'amplitude, le B0 et les ADC suivent ensemble cette orientation. Les bobines restent
dans leur repère de calibration : changer cet alignement adapte donc les courants.
La valeur initiale 180° reprend ton orientation préférée.

**Repère commun facultatif** : les paramètres `FRAME_*` ne changent que la représentation
commune après cet alignement. Ils ne changent pas le placement relatif aux bobines.
Changer seulement ce repère doit conserver les courants physiques.

Exemples : 90° = quart de tour ; −30° = rotation horaire ; un miroir X seul =
`ANGLE=0, MIRROR_X=True, MIRROR_Y=False`. Deux miroirs équivalent à une rotation de 180°.
Après une modification, relancer le notebook depuis cette cellule pour recalculer
les positions, les champs et les courants correspondants.

In [ ]:
# Alignement physique des données d'imagerie dans le repère des bobines ACDC.
ALIGNMENT_ANGLE_DEG=180.0
ALIGNMENT_MIRROR_X=False
ALIGNMENT_MIRROR_Y=False

# Changement de représentation commun, sans déplacement relatif aux bobines.
FRAME_ANGLE_DEG=0.0
FRAME_MIRROR_X=False
FRAME_MIRROR_Y=False

# Objet et relaxation issus des trois échos.
OBJECT_THRESHOLD_FRACTION=0.04
BACKGROUND_CORNER_M=0.085
B0_MODE="regularized"          # "regularized" ou "raw".
B0_SIGMA_PIXELS=1.0             # Échelle native fixée, sans ajustement au scan EPI.
RATE_MODE="pooled"             # "pooled", "guarded" ou "none".
SUBSPINS_Z=8                   # 128² dans le plan, puis sous-spins dans les 5 mm.

GPU_PROFILE="auto"             # "cpu", "auto", "gtx1660ti", "v100", "blackwell".
GPU_PRECISION="auto"
GPU_MEMORY_FRACTION=0.35
SAMPLE_CHUNK=None
RECONSTRUCTION_CHUNK=256

MAX_CURRENT_A=3.0
TOTAL_CURRENT_LIMIT_A=30.0
FIELD_LIMIT_HZ=500.0
CURRENT_RESPONSE={"enabled":True,"tau_s":133e-6/np.log(100.)}
HARDWARE={"min_update_interval_s":8e-6,"dac_resolution_A":0.0006,
          "max_slew_A_per_s":None,"avg_abs_current_A":None,
          "coil_R_ohm":None,"coil_L_H":None,"max_voltage_V":None,"max_coil_power_W":None}

IMAGE_INITIALIZATION="zero"  # "static" ou "zero".
MAX_ITERATIONS=100
LEARNING_RATE_A=0.01
PATIENCE=40
TARGET_MSE=None
LIVE_UPDATE_EVERY=20
RUN_OPTIMIZATION=True

# Export local seulement ; le contrôleur est utilisé séparément.
EXPORT_WAVEFORM=True
CSV_TIME_ORIGIN="first_imaging_trigger"  # Ou "rf".
TRIGGER_CHANNEL="ext1"
WAVEFORM_PARAMETERS={"channel_map":list(range(1,33)),"channel_polarity":[1]*32,
    "max_current_A":MAX_CURRENT_A,"total_current_limit_A":TOTAL_CURRENT_LIMIT_A,
    "min_dac_interval_s":8e-6}

CALIBRATION_PARAMETERS={"units":"Hz/A","registration_confirmed":True,
    "patient_to_source_m":np.eye(4).tolist(),
    "source_sha256":"bace6a584e780d80af8f2ca7c12957bd5b9dc7462be0968f949cc9a99243a06c",
    "registration_note":"Placement Bay5 confirmé précédemment ; orientation d'imagerie réglée ci-dessus."}
CORRECTED_SCAN_NAMES=["meas_MID00085_FID46892_scan.dat","meas_MID00086_FID46893_scan.dat"]

In [ ]:
def table(rows,headings=("Paramètre","Valeur")):
    display(Markdown("| "+" | ".join(headings)+" |\n| "+" | ".join(["---"]*len(headings))+" |\n"+
        "\n".join("| "+" | ".join(str(v) for v in row)+" |" for row in rows)))

def show(fig):
    plt.show();plt.close(fig)

def image_physical(ax,image,X,Y,title,cmap="gray",**options):
    artist=ax.pcolormesh(X*1000,Y*1000,np.asarray(image),shading="auto",cmap=cmap,**options)
    ax.set(title=title,xlabel="X [mm]",ylabel="Y [mm]",aspect="equal")
    return artist

def json_safe(value):
    if isinstance(value,dict):return {str(k):json_safe(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)):return [json_safe(v) for v in value]
    if isinstance(value,np.ndarray):return json_safe(value.tolist())
    if isinstance(value,np.generic):return json_safe(value.item())
    if isinstance(value,Path):return str(value)
    if isinstance(value,float) and not np.isfinite(value):return None
    return value

angles={"alignment_angle_deg":ALIGNMENT_ANGLE_DEG,"alignment_mirror_x":ALIGNMENT_MIRROR_X,
    "alignment_mirror_y":ALIGNMENT_MIRROR_Y,"frame_angle_deg":FRAME_ANGLE_DEG,
    "frame_mirror_x":FRAME_MIRROR_X,"frame_mirror_y":FRAME_MIRROR_Y}
Q=inplane_matrix(ALIGNMENT_ANGLE_DEG,ALIGNMENT_MIRROR_X,ALIGNMENT_MIRROR_Y)
P=inplane_matrix(FRAME_ANGLE_DEG,FRAME_MIRROR_X,FRAME_MIRROR_Y)
O=P@Q
table(list(angles.items()))
print("Matrice données→bobines Q :\n",Q,"\nMatrice du repère de calcul O=P@Q :\n",O)
print("Sorties :",OUTPUT_DIR)

## 2. Lire les scans et caractériser les trois échos

Les lecteurs conservent l'ordre ADC et la correction des navigateurs. Les images
ne sont pas tournées par `rot90`. L'amplitude native reste une amplitude GRE effective,
avec saturation et réception ; les trois échos ne déterminent pas T1.
Les ajustements locaux peu fiables utilisent un repli explicite sans décroissance
supplémentaire. Le taux global conservateur est le choix retenu par la V4.

In [ ]:
inputs=read_twin_inputs(INPUT_DIR)
fit=prepare_echo_object(inputs,OBJECT_THRESHOLD_FRACTION,BACKGROUND_CORNER_M)
raw=inputs["b0_raw"]
acquisition_native=inputs["acquisition"]
print("Scan sans ACDC :",inputs["paths"]["scan"].name)
table([("TE des trois échos [ms]",inputs["echo_times_s"]*1000),
       ("TE EPI [ms]",acquisition_native["TE_s"]*1000),
       ("Taux global effectif R2* [s⁻¹]",fit["pooled"]["R2star_sInv"]),
       ("Incertitude du taux global [s⁻¹]",fit["pooled"]["R2star_std_sInv"]),
       ("Ajustements locaux utilisables à TE EPI",int(fit["target_reliable_mask"].sum()))])
fig,axes=plt.subplots(1,3,figsize=(14,4),layout="constrained")
for ax,image,title in zip(axes,[fit["first_echo_amplitude"],inputs["field"]["b0Hz"],fit["target_reliable_mask"]],
                         ["Amplitude native 128²","B0 brut [Hz]","Fiabilité des ajustements locaux"]):
    im=ax.imshow(image,origin="lower",cmap="RdBu_r" if "B0" in title else "gray")
    ax.set_title(title);fig.colorbar(im,ax=ax,shrink=.8)
show(fig)

## 3. Appliquer rotation et miroirs, puis échantillonner les bobines 3D

Les valeurs d'amplitude et de B0 restent attachées aux mêmes positions natives.
Q transforme leur repère d'imagerie vers les bobines. P transforme ensuite le repère
commun des spins, du k-space et de la reconstruction. Les ADC ne sont ni interpolés,
ni réordonnés ni supprimés.

Chaque pixel natif possède huit positions suivant l'épaisseur. Les réponses ACDC sont
interpolées dans le volume 3D à chacune de ces positions. Un parent incomplet est exclu
sans renormalisation ; la fraction de signal perdue est affichée. Le domaine simulé
est indépendant d'une ROI d'optimisation. Les contraintes de champ portent par défaut
sur **tous les sous-spins conservés**.

In [ ]:
prepared=prepare_echo_shim_inputs(inputs,fit,INPUT_DIR/"ACDC_3T.npz",CALIBRATION_PARAMETERS,
    **angles,subspins_z=SUBSPINS_Z,rate_mode=RATE_MODE,b0_mode=B0_MODE,
    b0_sigma_pixels=B0_SIGMA_PIXELS,current_response=CURRENT_RESPONSE)
experiment=prepared["expanded_experiment"]
calibration=prepared["expanded_calibration"]
preparation_report=prepared["report"]
assert np.allclose(prepared["combined_frame_matrix"],O,rtol=0,atol=1e-12)
assert preparation_report["all_children_in_field_limits"]
table([(key,preparation_report[key]) for key in ["requested_parent_count","baseline_B0_parent_count",
    "retained_parent_count","subspin_count","excluded_missing_B0_amplitude_fraction",
    "excluded_ACDC_amplitude_fraction_of_baseline","amplitude_area_ratio","slice_thickness_mm"]])
print("Miroir / rotation : aucune interpolation des tableaux d'image.")

fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
image_physical(axes[0],prepared["native_amplitude"],prepared["physical_X_m"],prepared["physical_Y_m"],
               "Objet dans le repère des bobines")
image_physical(axes[1],prepared["native_B0_Hz"],prepared["physical_X_m"],prepared["physical_Y_m"],
               "B0 dans le repère des bobines [Hz]",cmap="RdBu_r")
image_physical(axes[2],prepared["parent_valid_mask"],prepared["physical_X_m"],prepared["physical_Y_m"],
               "Couverture des huit sous-spins")
for column,color,label in [(0,"tab:red","+X données"),(1,"tab:blue","+Y données")]:
    axes[0].arrow(0,0,25*Q[0,column],25*Q[1,column],color=color,width=.8,length_includes_head=True)
    axes[0].text(30*Q[0,column],30*Q[1,column],label,color=color,fontsize=8)
show(fig)

## 4. Construire MRzero avec les paramètres physiques du 04

L'amplitude est ancrée au premier TE GRE mesuré, puis évolue jusqu'aux ADC EPI.
Les poids d'aire 128²→64² et les fractions de volume suivant z sont appliqués une seule fois.
Le B0 suivant z reste inconnu ; il est constant dans la coupe, tandis que les cartes ACDC
varient réellement avec z. La référence B0 nul conserve cet objet et cette relaxation.

La réponse matérielle et son intégrale de phase restent continues entre les lignes.
Le solveur utilise l'adjoint exact de la reconstruction dans le repère choisi.

In [ ]:
def reconstruction_factory(acquisition,device,dtype):
    return FrameReconstruction(acquisition,device,dtype,frame_matrix=O,
                               sample_chunk=RECONSTRUCTION_CHUNK)

model=EchoSubspinACDCModel(experiment,calibration,max_current_A=MAX_CURRENT_A,
    hardware=HARDWARE,reconstruction_factory=reconstruction_factory,
    profile=GPU_PROFILE,precision=GPU_PRECISION,memory_fraction=GPU_MEMORY_FRACTION,
    sample_chunk=SAMPLE_CHUNK)
X_image,Y_image=model.reconstruction.X_m,model.reconstruction.Y_m
measured_coils=model.reconstruction.numpy_image(
    np.moveaxis(inputs["scan_diagnostics"]["corrected_kspace"],1,0))
measured_image=np.sqrt(np.sum(abs(measured_coils)**2,axis=0))
zero_currents=np.zeros((model.nc,model.nstates))
ideal_image=model.ideal_image
baseline_image,baseline_kdata=model.forward(zero_currents)
baseline_model_mse=float(np.sum((abs(baseline_image)-abs(ideal_image))**2)/np.sum(abs(ideal_image)**2))
baseline_measured_metrics=compare_magnitude(measured_image,baseline_image,X_m=X_image,Y_m=Y_image)
table([("Spins simulés",int(model.support.sum())),("Cache MRzero",model.runtime_info["native_baseline_cache"]),
       ("MSE sans ACDC / cible simulée",baseline_model_mse),
       ("NRMSE du twin / EPI réelle [%]",baseline_measured_metrics["NRMSE_percent"])])
fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
gain=baseline_measured_metrics["gain"]
for ax,image,title in zip(axes,[gain*abs(ideal_image),gain*abs(baseline_image),measured_image],
                         ["MRzero : B0 nul","MRzero : B0 mesuré","EPI réelle sans ACDC"]):
    image_physical(ax,image,X_image,Y_image,title,vmin=0,vmax=measured_image.max()*1.1)
show(fig)

### Vérifier le raccord au modèle V4

À courant nul et kz=0, les huit positions suivant z ont ici le même B0 et la même
amplitude par fraction de volume. Leur somme doit retrouver la simulation native128
sur les mêmes parents. Ce contrôle vérifie l'ancrage temporel, les poids et le repère.

In [ ]:
reference_check=NativeEPITwin(experiment["acquisition"],prepared["frame_X_m"],prepared["frame_Y_m"],
    prepared["native_amplitude"]*preparation_report["amplitude_area_ratio"],prepared["native_B0_Hz"],
    prepared["parent_valid_mask"],rate_sInv=prepared["native_rate_sInv"],
    amplitude_reference_time_s=fit["amplitude_reference_time_s"],device=str(model.device),sample_chunk=128)
# Le signal natif est comparé avant reconstruction, indépendante de l'ordre d'affichage.
reference_result=reference_check.simulate(b0_enabled=True)
zero_current_parity=float(np.linalg.norm(reference_result["kdata"][0]-baseline_kdata)/np.linalg.norm(baseline_kdata))
assert zero_current_parity<1e-4
print("Erreur relative V4 / modèle complet à courant nul :",zero_current_parity)
del reference_check,reference_result
if model.device.type=="cuda":torch.cuda.empty_cache()

## 5. Initialisation par shim statique et contrôle matériel

La solution statique fournit un point de départ à l'optimisation d'image. Tous les
sous-spins participent aux limites de champ. Les valeurs électriques non mesurées
restent désactivées ; aucun paramètre R/L ou thermique n'est inventé.

In [ ]:
if IMAGE_INITIALIZATION=="static":
    static_fit=solve_field_gpu(model.maps_numpy[model.quality_mask],
        -experiment["field"]["b0Hz"][model.quality_mask],device=model.device,dtype=model.dtype,
        max_current_A=MAX_CURRENT_A,total_current_limit_A=TOTAL_CURRENT_LIMIT_A,
        field_limit_Hz=FIELD_LIMIT_HZ,max_iterations=1000,tolerance=1e-7)
    requested_initial=np.repeat(static_fit["currents_A"][:,:1],model.nstates,axis=1)
elif IMAGE_INITIALIZATION=="zero":requested_initial=zero_currents.copy()
else:raise ValueError("IMAGE_INITIALIZATION doit valoir static ou zero.")
initial_currents=model.project_currents(requested_initial,TOTAL_CURRENT_LIMIT_A,FIELD_LIMIT_HZ)
initial_hardware=model.hardware.diagnostics(initial_currents,TOTAL_CURRENT_LIMIT_A,FIELD_LIMIT_HZ)
assert initial_hardware["all_limits_met"]
initial_image,_=model.forward(initial_currents)
table(list(initial_hardware["checks"].items()),("Contrôle initial","Résultat"))
print("Réponse équivalente : tau [µs] =",model.current_response["tau_s"][:3]*1e6)

## 6. Optimisation des courants avec affichage dynamique

La cible est le jumeau EPI à B0 nul, calculé avec le même objet et la même relaxation.
Le modèle évalué utilise B0 et chaque réponse ACDC à chaque sous-spin avant la somme
complexe. L'orientation réglée au début intervient donc dans tous les gradients.
L'affichage dynamique conserve l'ordre des pixels acquis ; les figures physiques
utilisent les coordonnées transformées. Les graphes CUDA du 03 sont désactivés pour
conserver la projection matérielle du 04.

In [ ]:
solution=None
if RUN_OPTIMIZATION:
    live=LiveOptimizationPlot(reference=ideal_image,baseline=baseline_image,current_limit_a=MAX_CURRENT_A,
        title=f"05 : alignement {ALIGNMENT_ANGLE_DEG}°, miroirs X={ALIGNMENT_MIRROR_X}, Y={ALIGNMENT_MIRROR_Y}")
    def update(snapshot):
        live.set_history(snapshot["history"]["iteration"],snapshot["history"]["fval"])
        live.update(iteration=snapshot["iteration"],image=snapshot["image"],currents_a=snapshot["currents_A"],
            loss=snapshot["fval"],nrmse_percent=100*np.sqrt(snapshot["fval"]),
            max_field_hz=snapshot["max_field_Hz"],field_limit_hz=FIELD_LIMIT_HZ)
    try:
        solution=optimize_mrzero_image(model,initial_currents,total_current_limit_A=TOTAL_CURRENT_LIMIT_A,
            field_limit_Hz=FIELD_LIMIT_HZ,max_iterations=MAX_ITERATIONS,learning_rate_A=LEARNING_RATE_A,
            target_mse=TARGET_MSE,patience=PATIENCE,callback=update,callback_every=LIVE_UPDATE_EVERY,
            use_cuda_graph=False)
    finally:
        live.finish("Terminé" if solution is not None else "Arrêt : dernier état évalué")
    info=solution["optimization_info"]
    currents=solution["currents_A"]
    history=solution["history"]
    table(list(info.items()))
else:print("Optimisation désactivée ; les diagnostics du modèle restent disponibles.")

## 7. Vérifier les courants physiques et la conversion DAC

On recalcule les limites entre les ADC, sur le TR complet, puis après conversion
nominale DAC. La MSE quantifiée doit aussi rester meilleure que celle du point initial.
Une limite d'itérations atteinte est rapportée comme telle, sans prétendre à une
convergence mathématique.

In [ ]:
solver_ok=False
if solution is not None:
    hardware_report=model.hardware.diagnostics(currents,TOTAL_CURRENT_LIMIT_A,FIELD_LIMIT_HZ)
    quantized_currents=quantize_currents(currents)
    quantized_hardware=model.hardware.diagnostics(quantized_currents,TOTAL_CURRENT_LIMIT_A,FIELD_LIMIT_HZ)
    quantized_image,quantized_kdata=model.forward(quantized_currents)
    quantized_mse=float(np.sum((abs(quantized_image)-abs(ideal_image))**2)/np.sum(abs(ideal_image)**2))
    solver_ok=bool(hardware_report["all_limits_met"] and quantized_hardware["all_limits_met"]
        and np.isfinite(info["fval"]) and info["fval"]<=solution["initial_relative_mse"]+1e-7
        and np.isfinite(quantized_mse) and quantized_mse<=solution["initial_relative_mse"]+1e-7)
    table([(key,value,quantized_hardware["checks"][key]) for key,value in hardware_report["checks"].items()],
          ("Contrôle matériel","Optimisé","Après DAC"))
    table([("MSE point initial",solution["initial_relative_mse"]),("MSE optimisée",info["fval"]),
           ("MSE après DAC",quantized_mse),("Contrôles numériques d'export",solver_ok)])
    fig,axes=plt.subplots(1,4,figsize=(18,4),layout="constrained")
    vmax=max(abs(image).max() for image in (ideal_image,baseline_image,solution["image"],quantized_image))
    for ax,image,title in zip(axes,[ideal_image,baseline_image,solution["image"],quantized_image],
                             ["Cible EPI B0 nul","Sans ACDC","ACDC optimisé","Après DAC"]):
        image_physical(ax,abs(image),X_image,Y_image,title,vmin=0,vmax=vmax)
    show(fig)
    fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
    im=axes[0].imshow(currents,cmap="RdBu_r",aspect="auto",vmin=-MAX_CURRENT_A,vmax=MAX_CURRENT_A)
    fig.colorbar(im,ax=axes[0]);axes[0].set(title="Commandes DAC",xlabel="Ligne chronologique",ylabel="Canal")
    axes[1].plot(history["iteration"],history["fval"],label="Évalué")
    axes[1].plot(history["iteration"],history["best_fval"],label="Meilleur")
    axes[1].set(title="MSE relative",xlabel="Itération");axes[1].legend()
    axes[2].bar(np.arange(model.nc)+1,hardware_report["avg_abs_current_A"])
    axes[2].set(title="Moyenne du courant physique sur le TR",xlabel="Canal",ylabel="Moyenne |I| [A]")
    show(fig)

## 8. Réponse temporelle et champs des sous-spins

La comparaison instantanée/filtrée utilise exactement les mêmes commandes. Les cartes
parentales ci-dessous sont des moyennes pour l'affichage ; les phases et les limites
ont été calculées sur chaque enfant.

In [ ]:
if solution is not None:
    response=solution["response_diagnostics"]
    instantaneous_image,_=model.forward(currents,response_enabled=False)
    fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
    for ax,image,title in zip(axes,[instantaneous_image,solution["image"],abs(instantaneous_image)-abs(solution["image"])],
                             ["Même commande : réponse instantanée","Réponse filtrée 133 µs","Différence de magnitude"]):
        image_physical(ax,abs(image) if "Différence" not in title else image,X_image,Y_image,title,
                       cmap="RdBu_r" if "Différence" in title else "gray")
    show(fig)
    field_parent=response["generated_field_at_line_center_Hz"]
    fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
    for ax,index in zip(axes,[0,model.nstates//2,model.nstates-1]):
        im=image_physical(ax,field_parent[...,index],prepared["physical_X_m"],prepared["physical_Y_m"],
                          f"Champ ACDC moyen : ligne {index+1} [Hz]",cmap="RdBu_r")
        fig.colorbar(im,ax=ax,shrink=.8)
    show(fig)

## 9. Comparaisons avec les acquisitions corrigées historiques

MID00085 et MID00086 utilisent d'anciennes formes d'onde. Leur affichage est un
contrôle des données ; ils ne valident pas les courants calculés dans ce notebook.
On conserve le gain obtenu avec l'EPI sans ACDC. La nouvelle solution reste une
prédiction jusqu'à l'acquisition du scan correspondant.

In [ ]:
# historical_images={}
# for filename in CORRECTED_SCAN_NAMES:
#     path=INPUT_DIR/"results"/filename
#     if not path.is_file():continue
#     _,metadata,diagnostics=read_scan(path,acquisition_native,return_intermediates=True)
#     assert np.allclose(metadata["slice_position_mm"],inputs["scan_metadata"]["slice_position_mm"],atol=1e-3)
#     coils=model.reconstruction.numpy_image(np.moveaxis(diagnostics["corrected_kspace"],1,0))
#     historical_images[filename]=np.sqrt(np.sum(abs(coils)**2,axis=0))
# fig,axes=plt.subplots(1,2+len(historical_images),figsize=(5*(2+len(historical_images)),4),layout="constrained")
# comparison=[("Cible simulée B0 nul",gain*abs(ideal_image)),("EPI sans ACDC",measured_image)]+list(historical_images.items())
# for ax,(label,image) in zip(axes,comparison):
#     image_physical(ax,image,X_image,Y_image,label,vmin=0,vmax=measured_image.max()*1.1)
# show(fig)

## 10. Sauvegarde complète de l'optimisation et des orientations

Les données enregistrées contiennent les matrices Q/P/O, les miroirs, les positions
patient des sous-spins, l'amplitude ancrée, le B0, les taux, les courants et les contrôles.
Les noms de session évitent d'écraser une optimisation précédente.

In [ ]:
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
arrays={"ideal_image":ideal_image,"baseline_image":baseline_image,"measured_image":measured_image,
    "initial_currents_A":initial_currents,"X_image_m":X_image,"Y_image_m":Y_image,
    "alignment_matrix":Q,"passive_frame_matrix":P,"combined_frame_matrix":O,
    "native_X_m":prepared["native_X_m"],"native_Y_m":prepared["native_Y_m"],
    "native_amplitude":prepared["native_amplitude"],"native_rate_sInv":prepared["native_rate_sInv"],
    "native_B0_Hz":prepared["native_B0_Hz"],"parent_valid_mask":prepared["parent_valid_mask"],
    "parent_flat_index":prepared["parent_flat_index"],"subspin_weights":prepared["weights"],
    "subspin_xyz_patient_m":prepared["xyz_m"],"subspin_B0_Hz":prepared["subspin_b0"],
    "subspin_basis_Hz_per_A":prepared["fieldbasis"],"subspin_amplitude_at_reference":experiment["m0_proxy"],
    "kx_mInv":experiment["acquisition"]["kx_mInv"],"ky_mInv":experiment["acquisition"]["ky_mInv"],
    "ADC_time_s":experiment["acquisition"]["time_s"]}
if solution is not None:
    arrays.update(optimized_image=solution["image"],optimized_kdata=solution["kdata"],currents_A=currents,
        quantized_currents_A=quantized_currents,quantized_image=quantized_image,
        iteration=history["iteration"],relative_mse=history["fval"],best_relative_mse=history["best_fval"])
np.savez_compressed(OUTPUT_DIR/"mrzero05_optimization.npz",**arrays)
source_hashes=dict(inputs["source_sha256"],calibration=preparation_report["calibration_source_sha256"])
summary={"input_dir":INPUT_DIR,"baseline_scan":inputs["paths"]["scan"].name,"orientation":angles,
    "alignment_matrix":Q,"passive_frame_matrix":P,"combined_frame_matrix":O,
    "preparation":preparation_report,"echo_fit":fit["report"],"runtime":model.runtime_info,
    "zero_current_V4_parity":zero_current_parity,"baseline_measured_comparison":baseline_measured_metrics,
    "baseline_vs_zeroB0_mse":baseline_model_mse,"current_limits_A":{"per_channel":MAX_CURRENT_A,"total":TOTAL_CURRENT_LIMIT_A},
    "current_response":CURRENT_RESPONSE,"hardware_configuration":HARDWARE,
    "source_sha256":source_hashes,"export_checks_passed":solver_ok,
    "historical_scans":list(historical_images),"historical_scans_use_current_optimisation":False}
if solution is not None:
    summary.update(optimization=info,hardware=hardware_report,hardware_after_DAC=quantized_hardware,
        quantized_relative_mse=quantized_mse,initial_relative_mse=solution["initial_relative_mse"])
(OUTPUT_DIR/"mrzero05_optimization.json").write_text(json.dumps(json_safe(summary),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
print("Résultats enregistrés :",OUTPUT_DIR)

In [ ]:
EXPORT_WAVEFORM=True

## 11. Export CSV facultatif, synchronisé à l'origine choisie

Le trigger ext1 du premier bloc d'imagerie est vérifié dans le fichier `.seq`.
En mode `first_imaging_trigger`, le temps du trigger est soustrait des temps RF
utilisés par la simulation. Le CSV conserve les temps en secondes et un retour à zéro.
Il contient des ampères, avec la correspondance canal/polarité choisie au début.
Le notebook produit un fichier local ; il ne programme pas le contrôleur.

In [ ]:
trigger_report=sequence_trigger_timing(inputs["paths"]["scan_sequence"],inputs["timing"],channel=TRIGGER_CHANNEL)
csv_edges,csv_timing=waveform_edges_for_origin(model,origin=CSV_TIME_ORIGIN,
    trigger_rf_s=trigger_report["trigger_rf_s"],min_update_interval_s=8e-6)
summary["trigger"]=trigger_report
summary["CSV_timing"]=csv_timing
print("Origine CSV :",CSV_TIME_ORIGIN,"| premier/dernier temps [ms] :",csv_edges[[0,-1]]*1000)
csv_file=None
if EXPORT_WAVEFORM and solution is not None and solver_ok:
    export=export_waveform(currents,csv_edges,WAVEFORM_PARAMETERS,OUTPUT_DIR,
        include_header=True,include_zero_preamble=True,include_final_zero_row=True,
        zero_time_column=False,numbered_filename=True)
    csv_file=Path(export["csv_file"])
    summary["waveform_export"]=export
    print("CSV :",csv_file)
else:print("Export CSV désactivé, ou contrôles numériques non satisfaits.")
(OUTPUT_DIR/"mrzero05_optimization.json").write_text(json.dumps(json_safe(summary),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")